<a href="https://colab.research.google.com/github/shahwaiz-9/Deep-Learning/blob/main/MultiClass_using_LSTM_%2C_RNN_%2C_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-04-08 02:00:54--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-04-08 02:00:54--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-04-08 02:00:54--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [2]:
pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.6 MB/s eta 0:00:00


In [3]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.7 MB/s eta 0:00:00


In [4]:
from gensim.models import Word2Vec

In [5]:
import re
import contractions
from sklearn.datasets import fetch_20newsgroups

In [50]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SimpleRNN, Bidirectional, SpatialDropout1D, GRU
import numpy as np

In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import wordnet

In [8]:
# Download necessary NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [9]:
import re
import contractions
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split


In [10]:
# Fetch the entire 20 Newsgroups dataset
full_data = fetch_20newsgroups(
    subset='all', # Fetch all data at once
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

In [11]:
import pandas as pd

# Create a single DataFrame for the entire dataset
full_df = pd.DataFrame({'text': full_data.data, 'target': full_data.target})

print("Full DataFrame head:")
display(full_df.head())

Full DataFrame head:


,text,target
0,\n\nI am sure some bashers of Pens fans are pr...,10
1,My brother is in the market for a high-perform...,3
2,\n\n\n\n\tFinally you said what you dream abou...,17
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,3
4,1) I have an old Jasmine drive which I cann...,4


In [12]:
full_df['target'].value_counts()

,count
target,
10,999
15,997
8,996
9,994
11,991
7,990
13,990
5,988
14,987


In [14]:
def clean_text(text):
    text = contractions.fix(text)
    text = text.lower()
    text = re.sub(r'\S*@\S*\s?', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = [w for w in text.split() if len(w) > 1]
    return " ".join(words)



In [16]:

full_df['cleaned_text'] = full_df['text'].apply(clean_text)
full_df['cleaned_text'][0]

'am sure some bashers of pens fans are pretty confused about the lack of any kind of posts about the recent pens massacre of the devils actually am bit puzzled too and bit relieved however am going to put an end to non pittsburghers relief with bit of praise for the pens man they are killing those devils worse than thought jagr just showed you why he is much better than his regular season stats he is also lot fo fun to watch in the playoffs bowman should let jagr have lot of fun in the next couple of games since the pens are going to beat the pulp out of jersey anyway was very disappointed not to see the islanders lose the final regular season game pens rule'

In [19]:
stopwords = nltk.corpus.stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [17]:

def get_wordnet_pos(tag):
    """Convert NLTK POS tag to WordNet POS tag"""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default fallback

In [18]:
def tokenize_lemmitize(text):

  # Word tokenization
  tokens = word_tokenize(text)

  pos_tags = nltk.pos_tag(tokens)

  lemmatized = []

  for word, tag in pos_tags:
    wn_tag = get_wordnet_pos(tag)
    lemma = lemmatizer.lemmatize(word, pos=wn_tag)

    if word not in stopwords and len(lemma) > 1:
      lemmatized.append(lemma)

  return lemmatized

In [21]:
full_df['processed_text'] = full_df['cleaned_text'].apply(tokenize_lemmitize)

In [23]:
w2v_model = Word2Vec(
    sentences=full_df['processed_text'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

print(f"Word2Vec Vocabulary Size: {len(w2v_model.wv.key_to_index)}")

Word2Vec Vocabulary Size: 39367


In [25]:
# Create a dictionary which contains the word information about dataset
#   {shahwaiz: 3 ( Ouccrence of the word )}

tokenizer = Tokenizer()
tokenizer.fit_on_texts(full_df['cleaned_text'])

vocab_size = len(tokenizer.word_index) + 1
print(f"Tokenizer Vocabulary Size: {vocab_size}")

Tokenizer Vocabulary Size: 84799


In [26]:
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print("Embedding Matrix Shape:", embedding_matrix.shape)

Embedding Matrix Shape: (84799, 100)


In [27]:
max_len = 200

X_seq = tokenizer.texts_to_sequences(full_df['cleaned_text'])

X_padded = pad_sequences(X_seq, maxlen=max_len, padding='post')

X_train, X_test, y_train, y_test = train_test_split(X_padded, full_df['target'], test_size=0.2, random_state=42)

In [36]:
X_padded[100]

array([ 1410,   107, 10565,    56,     1,  8157,  2912, 46768,  2860,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,

In [43]:
model_final = Sequential([

    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=True,
              mask_zero=True),
    SpatialDropout1D(0.4),
    LSTM(128, dropout=0.2, return_sequences=True, use_cudnn=False),
    LSTM(64, dropout=0.2, use_cudnn=False),
    Dense(20, activation='softmax')
])

model_final.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

model_final.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │     8,479,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_6             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,479,900 (32.35 MB)

 Trainable params: 8,479,900 (32.35 MB)

 Non-trainable params: 0 (0.00 B)

In [44]:
history = model_final.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 25s 71ms/step - accuracy: 0.2127 - loss: 2.4013 - val_accuracy: 0.3034 - val_loss: 2.0655
Epoch 2/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.3286 - loss: 2.0245 - val_accuracy: 0.3958 - val_loss: 1.7712
Epoch 3/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.3959 - loss: 1.7968 - val_accuracy: 0.4225 - val_loss: 1.6773
Epoch 4/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.4430 - loss: 1.6385 - val_accuracy: 0.4546 - val_loss: 1.6161
Epoch 5/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 12s 49ms/step - accuracy: 0.4930 - loss: 1.4685 - val_accuracy: 0.4891 - val_loss: 1.5024
Epoch 6/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.5406 - loss: 1.3356 - val_accuracy: 0.5159 - val_loss: 1.4229
Epoch 7/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 21s 44ms/step - accuracy: 0.5928 - loss: 1.1944 - val_accuracy: 0.5509 - val_loss: 1.3731
Epoch 8/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - accuracy: 0.6217 - loss: 1.1036 - 

In [47]:
rnn_model = Sequential([

    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=True,
              mask_zero=True),
    SimpleRNN(128, dropout=0.2, return_sequences=True),
    SimpleRNN(64, dropout=0.2),
    Dense(20, activation='softmax')
])

rnn_model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

rnn_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ ?                      │     8,479,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,479,900 (32.35 MB)

 Trainable params: 8,479,900 (32.35 MB)

 Non-trainable params: 0 (0.00 B)

In [48]:
rnn_history = rnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 18s 55ms/step - accuracy: 0.1348 - loss: 2.7586 - val_accuracy: 0.2034 - val_loss: 2.4842
Epoch 2/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 13s 56ms/step - accuracy: 0.1767 - loss: 2.6254 - val_accuracy: 0.1570 - val_loss: 2.6312
Epoch 3/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.2149 - loss: 2.4683 - val_accuracy: 0.2188 - val_loss: 2.5175
Epoch 4/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - accuracy: 0.2098 - loss: 2.5403 - val_accuracy: 0.2154 - val_loss: 2.4584
Epoch 5/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.2455 - loss: 2.4108 - val_accuracy: 0.2438 - val_loss: 2.3680
Epoch 6/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.3130 - loss: 2.1752 - val_accuracy: 0.2655 - val_loss: 2.3194
Epoch 7/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.3595 - loss: 2.0595 - val_accuracy: 0.3064 - val_loss: 2.1975
Epoch 8/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step - accuracy: 0.3989 - loss: 1.9356 - val

In [56]:
gru_model = Sequential([
        Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=True,
              mask_zero=True),
        SpatialDropout1D(0.4),
        GRU(128, dropout=0.2, return_sequences=True, use_cudnn=False),
        GRU(64, dropout=0.2, use_cudnn=False),
        Dense(20, activation='softmax')
])


gru_model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

gru_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ ?                      │     8,479,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_11            │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,479,900 (32.35 MB)

 Trainable params: 8,479,900 (32.35 MB)

 Non-trainable params: 0 (0.00 B)

In [57]:
gru_history = gru_model.fit(X_train, y_train, epochs=15, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 30s 82ms/step - accuracy: 0.1867 - loss: 2.5368 - val_accuracy: 0.2947 - val_loss: 2.0339
Epoch 2/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 18s 75ms/step - accuracy: 0.3490 - loss: 1.9171 - val_accuracy: 0.4146 - val_loss: 1.6731
Epoch 3/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 14s 60ms/step - accuracy: 0.4507 - loss: 1.5939 - val_accuracy: 0.4682 - val_loss: 1.5408
Epoch 4/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 14s 60ms/step - accuracy: 0.5249 - loss: 1.3715 - val_accuracy: 0.5204 - val_loss: 1.4019
Epoch 5/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 22s 64ms/step - accuracy: 0.5972 - loss: 1.1739 - val_accuracy: 0.5775 - val_loss: 1.2925
Epoch 6/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 21s 65ms/step - accuracy: 0.6668 - loss: 0.9927 - val_accuracy: 0.6056 - val_loss: 1.2198
Epoch 7/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 20s 64ms/step - accuracy: 0.7227 - loss: 0.8182 - val_accuracy: 0.6228 - val_loss: 1.1995
Epoch 8/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 23s 73ms/step - accuracy: 0.7754 - loss: 0.6797 - 

In [62]:
bi_model_final = Sequential([

    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=True,
              mask_zero=True),
    SpatialDropout1D(0.4),
    Bidirectional(LSTM(128, dropout=0.2, use_cudnn=False)),
    Dense(20, activation='softmax')
])

bi_model_final.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

bi_model_final.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_16 (Embedding)        │ ?                      │     8,479,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_14            │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,479,900 (32.35 MB)

 Trainable params: 8,479,900 (32.35 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
bi_history = bi_model_final.fit(
    X_train, y_train,
    epochs=15,
    validation_data=(X_test, y_test)
)

Epoch 1/15
472/472 ━━━━━━━━━━━━━━━━━━━━ 502s 1s/step - accuracy: 0.2787 - loss: 2.1620 - val_accuracy: 0.3724 - val_loss: 1.8432
Epoch 2/15
472/472 ━━━━━━━━━━━━━━━━━━━━ 493s 1s/step - accuracy: 0.4191 - loss: 1.7221 - val_accuracy: 0.4459 - val_loss: 1.6384
Epoch 3/15
472/472 ━━━━━━━━━━━━━━━━━━━━ 509s 1s/step - accuracy: 0.5057 - loss: 1.4680 - val_accuracy: 0.5072 - val_loss: 1.4485
Epoch 4/15
472/472 ━━━━━━━━━━━━━━━━━━━━ 496s 1s/step - accuracy: 0.5947 - loss: 1.2128 - val_accuracy: 0.5613 - val_loss: 1.3173
Epoch 5/15
472/472 ━━━━━━━━━━━━━━━━━━━━ 503s 1s/step - accuracy: 0.6662 - loss: 1.0034 - val_accuracy: 0.6005 - val_loss: 1.2624
Epoch 6/15
472/472 ━━━━━━━━━━━━━━━━━━━━ 497s 1s/step - accuracy: 0.7270 - loss: 0.8392 - val_accuracy: 0.6236 - val_loss: 1.2203
Epoch 7/15
 12/472 ━━━━━━━━━━━━━━━━━━━━ 8:04 1s/step - accuracy: 0.7768 - loss: 0.6497

In [ ]:
import matplotlib.pyplot as plt

# bar chart of all models accuracy
plt.figure(figsize=(10, 6))

plt.bar(['LSTM', 'RNN', 'GRU', 'BiDirectional'],
 [max(history, key=lambda x: x['val_accuracy'])['val_accuracy'], max(rnn_history, key=lambda x: x['val_accuracy'])['val_accuracy'], max(gru_history, key=lambda x: x['val_accuracy'])['val_accuracy'], max(bi_history, key = lamda x: x['val_accuracy'])['val_accuracy']], color=['blue', 'green', 'red', 'orange'] )
plt.xlabel('Models')
plt.ylabel('Validation Accuracy')

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(['LSTM', 'RNN', 'GRU', 'BiDirectional'],
 [max(history, key=lambda x: x['val_accuracy'])['val_accuracy'], max(rnn_history, key=lambda x: x['val_accuracy'])['val_accuracy'], max(gru_history, key=lambda x: x['val_accuracy'])['val_accuracy'], max], color=['blue', 'green', 'red'] )
plt.xlabel('Models')
plt.ylabel('Validation Accuracy')

In [ ]:
# model = Sequential([
#     Embedding(input_dim=10000,
#               output_dim=100,
#               weights=[embedding_matrix],
#               trainable=False,
#               mask_zero=True),
#     Bidirectional(LSTM(64, dropout=0.2)),
#     Dense(64, activation='relu'),
#     Dropout(0.3),
#     Dense(20, activation='softmax') # MUST be 20 for Newsgroups
# ])

In [ ]:
# model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# model.fit(X_train_pad, y_train, validation_data=(X_test_pad, y_test), epochs=10)

Epoch 1/10


InvalidArgumentError: Graph execution error:

Detected at node sequential_1/bidirectional_1/forward_lstm_1/Assert/Assert defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/u

In [ ]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(full_df['cleaned_text'])

all_sequences = tokenizer.texts_to_sequences(full_df['cleaned_text'])git
X_padded = pad_sequences(all_sequences, maxlen=200, padding='post', truncating='post')

print("Shape of padded sequences:", X_padded.shape)

In [ ]:
embeddings_index = {}
# Using the 100D version to match your current architecture
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


In [ ]:
embedding_dim = 100
embedding_matrix = np.zeros((10000, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < 10000:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            # Words not found in the glove index will remain all-zeros (padding/unknown)
            embedding_matrix[i] = embedding_vector